# LifeMate — ASR (zaic Whisper) + LLM (Qwen3-4B-Instruct-2507) trên Kaggle

**Chuẩn bị (một lần):**
1. *Add Input* → upload dataset chứa `train+public_test.zip` (Kaggle tự giải nén).
2. *Add Input* → upload dataset chứa `zaic-20261102T164447Z-1-001.zip` (mô hình ASR, thư mục `zaic/`).
3. *Settings*: **Accelerator = GPU T4 x2** (hoặc P100), **Internet = On** (để tải `Qwen/Qwen3-4B-Instruct-2507` ~8 GB).
   Nếu đã thêm FunReason-MT làm Kaggle Model/Dataset thì có thể tắt Internet — script tự dò thư mục `qwen3`.
4. Sửa prompt ở ô **System prompt** bên dưới (dán prompt của bạn vào), rồi *Run All* hoặc *Save Version → Save & Run All*.

Kết quả: `/kaggle/working/submission.jsonl` (634 dòng) + `traces.jsonl` (transcript, output thô của LLM từng bước để debug).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, torch
print("torch", torch.__version__, "transformers", transformers.__version__)
from packaging.version import Version
if Version(transformers.__version__) < Version("4.51"):
    !pip install -q "transformers>=4.51,<5" accelerate

## System prompt
Dán prompt của bạn vào đây (giữ dòng `%%writefile` ở đầu). Ngữ cảnh người dùng và danh sách tool được script tự nối thêm.

In [ ]:
%%writefile system_prompt.txt
Bạn là LifeMate, trợ lý giọng nói tiếng Việt điều khiển các ứng dụng của người dùng bằng cách gọi công cụ (tools).
Yêu cầu của người dùng được nhận dạng từ giọng nói nên có thể sai chính tả, thiếu dấu hoặc lẫn từ tiếng Anh; hãy hiểu theo ý gần nhất.

Quy tắc:
1. Chỉ gọi các tool có trong danh sách, đúng tên, đúng tên tham số, đúng kiểu dữ liệu và enum. Số điện thoại luôn là chuỗi.
2. Tham số ID (device_id, contact_id, event_id, ...) phải lấy từ kết quả tool đã gọi hoặc từ lịch sử hội thoại. Không tự bịa ID, không điền tên vào trường ID. Nếu cần ID mà chưa có, hãy gọi tool tìm kiếm/liệt kê phù hợp trước (ví dụ contacts.search, smart_home.get_status).
3. Thời gian tương đối ("mai", "tối nay", "3 giờ chiều") tính theo current_time và timezone trong ngữ cảnh; dùng định dạng ISO 8601 có múi giờ, ví dụ 2026-11-17T15:00:00+07:00.
4. "Nhà" và "công ty/chỗ làm" lấy từ user_profile (home_address, work_address). Vị trí hiện tại lấy từ location.
5. Lịch sử hội thoại đã được thực hiện xong; không lặp lại các thao tác trong lịch sử. Chỉ xử lý yêu cầu cuối cùng của người dùng.
6. Yêu cầu có điều kiện ("nếu ... thì ...") phải kiểm tra bằng tool trước rồi mới quyết định làm hay không.
7. Nếu thiếu thông tin bắt buộc mà không thể lấy từ ngữ cảnh hoặc tool, gọi sys.ask_missing_info với tool_name là tool cần gọi và params là danh sách tên tham số bắt buộc còn thiếu. Phần nào đủ thông tin thì làm trước.
8. Nếu không có ứng dụng nào được cài có thể xử lý yêu cầu, không gọi tool nào, chỉ trả lời ngắn gọn.
9. Không gọi tool thừa: không gọi lại tool đã thành công, không gọi chỉ để xác nhận. Khi đã hoàn tất, trả lời ngắn gọn bằng tiếng Việt và không gọi thêm tool.
10. Văn bản tự do (nội dung tin nhắn, tiêu đề, ghi chú) giữ đúng ý và từ ngữ người dùng nói.
11. Số đọc bằng chữ phải đổi sang số: "năm mươi ngàn" = 50000, "1 triệu rưỡi" = 1500000, "hai mươi sáu độ" = 26. Số điện thoại đọc từng chữ số: "không chín năm không" = "0950".
12. Chỉ điền tham số tùy chọn khi người dùng nói rõ giá trị đó. Không tự thêm location (mặc định là thành phố của người dùng), label, repeat, note, room, type... nếu người dùng không nói.
13. Trước khi gọi sys.ask_missing_info, hãy tìm thông tin bằng tool đọc dữ liệu (calendar.list_events cho địa điểm/giờ của sự kiện, contacts.search cho người, bill_payment.list_bills cho hóa đơn, banking.list_beneficiaries cho người nhận chuyển khoản...). Chỉ hỏi khi tool không có thông tin.
14. Mỗi lượt chỉ gọi các tool có thể gọi ngay; nếu tham số phụ thuộc kết quả của tool khác thì gọi tool đó trước và chờ kết quả.

## Script suy luận
Ghi `lifemate_infer.py` (bản gốc ở `kaggle/lifemate_infer.py` trong repo).

In [ ]:
%%writefile lifemate_infer.py
"""LifeMate end-to-end inference for Kaggle: Whisper ASR (zaic) -> FunReason-MT tool calling -> zaic_tools.

Writes one submission line per sample: {"id", "tool_calls": env.get_tool_calls(), "state_hash": env.get_state_hash()}.
Every sample always gets a line, even when the model fails (the calls already made are kept).

    python lifemate_infer.py --split public_test --out submission.jsonl
    python lifemate_infer.py --split train --limit 100 --eval          # local score on Train (gold labels)
    python lifemate_infer.py --split train --limit 100 --eval --gold-transcript   # isolate the LLM from ASR errors
"""
from __future__ import annotations

import argparse
import copy
import gc
import json
import os
import random
import re
import sys
import time
import unicodedata
import wave
from collections import Counter
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.dont_write_bytecode = True  # Kaggle inputs are read-only; zaic_tools is imported from there.

DEFAULT_LLM = "Qwen/Qwen3-4B-Instruct-2507"
SEED = 20261027
SEARCH_ROOTS = ("/kaggle/input", "/kaggle/working", ".", "..")

DEFAULT_SYSTEM_PROMPT = """Bạn là LifeMate, trợ lý giọng nói tiếng Việt điều khiển các ứng dụng của người dùng bằng cách gọi công cụ (tools).
Yêu cầu của người dùng được nhận dạng từ giọng nói nên có thể sai chính tả, thiếu dấu hoặc lẫn từ tiếng Anh; hãy hiểu theo ý gần nhất.

Quy tắc:
1. Chỉ gọi các tool có trong danh sách, đúng tên, đúng tên tham số, đúng kiểu dữ liệu và enum. Số điện thoại luôn là chuỗi.
2. Tham số ID (device_id, contact_id, event_id, ...) phải lấy từ kết quả tool đã gọi hoặc từ lịch sử hội thoại. Không tự bịa ID, không điền tên vào trường ID. Nếu cần ID mà chưa có, hãy gọi tool tìm kiếm/liệt kê phù hợp trước (ví dụ contacts.search, smart_home.get_status).
3. Thời gian tương đối ("mai", "tối nay", "3 giờ chiều") tính theo current_time và timezone trong ngữ cảnh; dùng định dạng ISO 8601 có múi giờ, ví dụ 2026-11-17T15:00:00+07:00.
4. "Nhà" và "công ty/chỗ làm" lấy từ user_profile (home_address, work_address). Vị trí hiện tại lấy từ location.
5. Lịch sử hội thoại đã được thực hiện xong; không lặp lại các thao tác trong lịch sử. Chỉ xử lý yêu cầu cuối cùng của người dùng.
6. Yêu cầu có điều kiện ("nếu ... thì ...") phải kiểm tra bằng tool trước rồi mới quyết định làm hay không.
7. Nếu thiếu thông tin bắt buộc mà không thể lấy từ ngữ cảnh hoặc tool, gọi sys.ask_missing_info với tool_name là tool cần gọi và params là danh sách tên tham số bắt buộc còn thiếu. Phần nào đủ thông tin thì làm trước.
8. Nếu không có ứng dụng nào được cài có thể xử lý yêu cầu, không gọi tool nào, chỉ trả lời ngắn gọn.
9. Không gọi tool thừa: không gọi lại tool đã thành công, không gọi chỉ để xác nhận. Khi đã hoàn tất, trả lời ngắn gọn bằng tiếng Việt và không gọi thêm tool.
10. Văn bản tự do (nội dung tin nhắn, tiêu đề, ghi chú) giữ đúng ý và từ ngữ người dùng nói.
11. Số đọc bằng chữ phải đổi sang số: "năm mươi ngàn" = 50000, "1 triệu rưỡi" = 1500000, "hai mươi sáu độ" = 26. Số điện thoại đọc từng chữ số: "không chín năm không" = "0950".
12. Chỉ điền tham số tùy chọn khi người dùng nói rõ giá trị đó. Không tự thêm location (mặc định là thành phố của người dùng), label, repeat, note, room, type... nếu người dùng không nói.
13. Trước khi gọi sys.ask_missing_info, hãy tìm thông tin bằng tool đọc dữ liệu (calendar.list_events cho địa điểm/giờ của sự kiện, contacts.search cho người, bill_payment.list_bills cho hóa đơn, banking.list_beneficiaries cho người nhận chuyển khoản...). Chỉ hỏi khi tool không có thông tin.
14. Mỗi lượt chỉ gọi các tool có thể gọi ngay; nếu tham số phụ thuộc kết quả của tool khác thì gọi tool đó trước và chờ kết quả."""


# ----------------------------------------------------------------------------------------------- discovery
def _walk(root: Path, max_depth: int):
    root = Path(root)
    if not root.is_dir():
        return
    stack = [(root, 0)]
    while stack:
        d, depth = stack.pop()
        yield d
        if depth >= max_depth:
            continue
        try:
            children = sorted(p for p in d.iterdir() if p.is_dir() and p.name != "__MACOSX" and not p.name.startswith("."))
        except OSError:
            continue
        # skip huge folders that never hold what we look for
        stack.extend((c, depth + 1) for c in reversed(children) if c.name not in {"audio"} and not c.name.endswith("_environment"))


def find_data_root(split: str, explicit: str | None) -> Path:
    if explicit:
        return Path(explicit).resolve()
    for base in SEARCH_ROOTS:
        for d in _walk(Path(base), 5):
            if (d / split / f"{split}.jsonl").is_file() and (d / split / "zaic_tools").is_dir():
                return d.resolve()
    raise SystemExit(f"Không tìm thấy thư mục dữ liệu chứa {split}/{split}.jsonl; truyền --data-root")


def _model_dirs(model_type: str):
    for base in SEARCH_ROOTS:
        for d in _walk(Path(base), 6):
            cfg = d / "config.json"
            if cfg.is_file():
                try:
                    if json.load(open(cfg, encoding="utf-8")).get("model_type") == model_type:
                        yield d.resolve()
                except (OSError, ValueError):
                    pass


def find_asr_model(explicit: str | None) -> Path:
    if explicit:
        return Path(explicit).resolve()
    for d in _model_dirs("whisper"):
        if (d / "preprocessor_config.json").is_file():
            return d
    raise SystemExit("Không tìm thấy mô hình Whisper (thư mục zaic/); truyền --asr-model")


def resolve_llm(spec: str, download_dir: str) -> Path:
    p = Path(spec)
    if p.is_dir() and (p / "config.json").is_file():
        return p.resolve()
    want = spec.split("/")[-1].lower()
    for d in _model_dirs("qwen3"):  # a Kaggle dataset/model of the same name already attached
        if any(d.glob("*.safetensors")) and want in str(d).lower():
            print(f"[llm] dùng bản có sẵn: {d}", flush=True)
            return d
    download_dir = download_dir or f"/tmp/{spec.split('/')[-1]}"
    if Path(download_dir, "config.json").is_file() and any(Path(download_dir).glob("*.safetensors")):
        return Path(download_dir)
    from huggingface_hub import snapshot_download
    print(f"[llm] tải {spec} về {download_dir} ...", flush=True)
    return Path(snapshot_download(spec, local_dir=download_dir,
                                  allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"]))


def seed_everything(seed: int = SEED):
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def gpu_dtype():
    import torch
    if not torch.cuda.is_available():
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16


# ----------------------------------------------------------------------------------------------- ASR
def load_wav(path: Path):
    import numpy as np
    try:
        import soundfile as sf
        audio, sr = sf.read(str(path), dtype="float32", always_2d=False)
    except Exception:
        with wave.open(str(path), "rb") as w:
            sr, ch, width = w.getframerate(), w.getnchannels(), w.getsampwidth()
            raw = w.readframes(w.getnframes())
        dtype = {1: np.uint8, 2: "<i2", 4: "<i4"}[width]
        audio = np.frombuffer(raw, dtype=dtype).astype(np.float32)
        audio = (audio - 128) / 128 if width == 1 else audio / float(2 ** (8 * width - 1))
        if ch > 1:
            audio = audio.reshape(-1, ch)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000 and len(audio):
        n = int(round(len(audio) * 16000 / sr))
        audio = np.interp(np.linspace(0, len(audio) - 1, n), np.arange(len(audio)), audio).astype(np.float32)
    return audio


class ASR:
    def __init__(self, model_dir: Path, device: str):
        import torch
        from transformers import WhisperForConditionalGeneration, WhisperProcessor
        self.torch, self.device = torch, device
        self.dtype = torch.float16 if device.startswith("cuda") else torch.float32
        self.processor = WhisperProcessor.from_pretrained(model_dir)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_dir, torch_dtype=self.dtype).to(device).eval()
        self.params = sum(p.numel() for p in self.model.parameters())
        print(f"[asr] {model_dir} params={self.params / 1e6:.0f}M dtype={self.dtype}", flush=True)

    def transcribe(self, clips, batch_size: int = 16) -> list[str]:
        """clips: list of float32 16 kHz arrays. Clips over 30 s are split into 30 s chunks and joined."""
        import numpy as np
        pieces, owner = [], []
        for i, a in enumerate(clips):
            chunks = [a[s:s + 480000] for s in range(0, max(len(a), 1), 480000)] or [a]
            for c in chunks:
                pieces.append(c if len(c) else np.zeros(1600, dtype=np.float32))
                owner.append(i)
        order = sorted(range(len(pieces)), key=lambda k: len(pieces[k]))
        out = [""] * len(pieces)
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            feats = self.processor([pieces[k] for k in idx], sampling_rate=16000, return_tensors="pt").input_features
            with self.torch.inference_mode():
                ids = self.model.generate(feats.to(self.device, self.dtype), language="vi", task="transcribe",
                                          do_sample=False, num_beams=1, max_new_tokens=440)
            for k, text in zip(idx, self.processor.batch_decode(ids, skip_special_tokens=True)):
                out[k] = text.strip()
        texts = [""] * len(clips)
        for k, i in enumerate(owner):
            texts[i] = (texts[i] + " " + out[k]).strip()
        return texts

    def close(self):
        del self.model
        gc.collect()
        if self.torch.cuda.is_available():
            self.torch.cuda.empty_cache()


def user_audio_paths(item) -> list[str]:
    paths = [t["audio"] for t in item.get("history", []) if t.get("role") == "user" and t.get("audio")]
    return paths + [item["audio"]]


def build_transcripts(items, split_dir: Path, asr_dir: Path, cache_path: Path, device: str, batch_size: int) -> dict:
    cache = json.load(open(cache_path, encoding="utf-8")) if cache_path.is_file() else {}
    todo = sorted({p for it in items for p in user_audio_paths(it)} - set(cache))
    if todo:
        asr = ASR(asr_dir, device)
        t0 = time.time()
        step = 256
        for s in range(0, len(todo), step):
            part = todo[s:s + step]
            clips = [load_wav(split_dir / p) for p in part]
            for p, text in zip(part, asr.transcribe(clips, batch_size)):
                cache[p] = text
            json.dump(cache, open(cache_path, "w", encoding="utf-8"), ensure_ascii=False)
            print(f"[asr] {min(s + step, len(todo))}/{len(todo)} clips ({time.time() - t0:.0f}s)", flush=True)
        asr.close()
    return cache


# ----------------------------------------------------------------------------------------------- tools/prompt
def load_catalog(split_dir: Path, split: str) -> dict:
    cat = json.load(open(split_dir / f"{split}_tools_catalog.json", encoding="utf-8"))
    return {t["name"]: t for app in cat["apps"] for t in app["tools"]}


def _clean_schema(node):
    if isinstance(node, dict):
        out = {}
        for k, v in node.items():
            if k == "x-match":
                continue
            if k == "x-ref":
                continue
            if k == "x-one-of":
                continue
            out[k] = _clean_schema(v)
        ref = node.get("x-ref")
        if ref and "description" not in out:
            out["description"] = f"ID lấy từ {ref}"
        return out
    if isinstance(node, list):
        return [_clean_schema(x) for x in node]
    return node


def tool_specs(catalog: dict, item) -> list[dict]:
    names = [n for app in item["context"]["installed_apps"].values() for n in app.get("tools", [])]
    names.append("sys.ask_missing_info")
    specs, seen = [], set()
    for n in names:
        if n in seen or n not in catalog:
            continue
        seen.add(n)
        t = catalog[n]
        params = _clean_schema(t.get("parameters") or {"type": "object", "properties": {}})
        groups = (t.get("parameters") or {}).get("x-one-of")
        desc = t.get("description", "")
        if groups and "một trong" not in desc:
            desc += ". Bắt buộc một trong: " + " hoặc ".join("+".join(g) for g in groups)
        if t.get("returns"):
            desc += f". Trả về: {t['returns']}"
        specs.append({"type": "function", "function": {"name": n, "description": desc, "parameters": params}})
    return specs


def dumps(v) -> str:
    return json.dumps(v, ensure_ascii=False, separators=(",", ":"))


def build_messages(item, system_prompt: str, transcripts: dict, gold_text: bool) -> list[dict]:
    ctx = {k: v for k, v in item["context"].items() if k != "installed_apps"}
    ctx["installed_apps"] = list(item["context"]["installed_apps"])
    system = system_prompt.strip() + "\n\n# Ngữ cảnh người dùng\n" + dumps(ctx)
    msgs = [{"role": "system", "content": system}]
    for turn in item.get("history", []):
        if turn.get("role") == "user":
            text = turn.get("audio_text") if gold_text and turn.get("audio_text") else transcripts.get(turn.get("audio"), "")
            msgs.append({"role": "user", "content": text or "(không nghe rõ)"})
        else:
            calls = turn.get("tool_calls") or []
            if calls:
                msgs.append({"role": "assistant", "content": "", "tool_calls": [
                    {"type": "function", "function": {"name": c["name"], "arguments": c.get("arguments") or {}}} for c in calls]})
                results = turn.get("tool_results") or []
                for i in range(len(calls)):
                    msgs.append({"role": "tool", "content": dumps(results[i]) if i < len(results) else '{"ok":true}'})
            msgs.append({"role": "assistant", "content": turn.get("text") or "Đã xong."})
    current = item.get("audio_text") if gold_text and item.get("audio_text") else transcripts.get(item["audio"], "")
    msgs.append({"role": "user", "content": current or "(không nghe rõ)"})
    return msgs


# ----------------------------------------------------------------------------------------------- parsing
_TOOL_CALL = re.compile(r"<tool_call>\s*(.*?)\s*(?:</tool_call>|$)", re.S)


def _loads_loose(s: str):
    s = s.strip()
    s = re.sub(r"^```(?:json)?|```$", "", s).strip()
    try:
        return json.loads(s)
    except ValueError:
        pass
    m = re.search(r"\{.*\}", s, re.S)
    if m:
        frag = re.sub(r",\s*([}\]])", r"\1", m.group(0))
        try:
            return json.loads(frag)
        except ValueError:
            pass
    return None


# FunReason-MT was trained on BFCL prompting mode, so after </think> it rarely uses <tool_call>{json}</tool_call>.
# Formats seen on Kaggle (v4/v6 traces):  <alarm.create time="..." repeat="once"/>   [event_tickets.book(event_id="e5")]
#   /sys.ask_missing_info(tool_name="x")   <food_delivery.cancel_order(order_id="o1")>   {"name": ..., "arguments": ...}
#   <function name="public_transit.topup_card" arguments={"amount": 150000}>
_TOOL_NAME = r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*"
_FUNC_TAG = re.compile(r"<function\s+name\s*=\s*[\"']?(" + _TOOL_NAME + r")[\"']?\s*(?:arguments|parameters)\s*=\s*"
                       r"(\{.*?\}|\"[^\"]*\"|'[^']*')\s*/?>", re.S)
_FUNC_NAME_TAG = re.compile(r"<(?:function_name|function-name|name|tool_name|function)>\s*(" + _TOOL_NAME + r")\s*"
                            r"</(?:function_name|function-name|name|tool_name|function)>\s*"
                            r"(?:<(?:arguments|parameters|args)>\s*(.*?)\s*(?:</(?:arguments|parameters|args)>|$))?", re.S)
_CHILD_TAG = re.compile(r"<([A-Za-z_][A-Za-z0-9_]*)>\s*(.*?)\s*</\1>", re.S)
_PROSE_HEAD = re.compile(r"^(?:\s*<(?:answer|response|reply|tool_response|result|function[ _-]?calls?)>)*\s*", re.I)


def looks_like_prose(head: str) -> bool:
    """True when the first tokens of an answer are plain text (closing reply or restarted reasoning), not a call."""
    rest = _PROSE_HEAD.sub("", head)
    if not rest.strip():
        return False
    if rest.lstrip()[:1] in "<[/{":
        return False
    return not re.search(_TOOL_NAME + r"\s*\(", rest)


def _args_block(raw: str) -> dict:
    """Argument block in any of: JSON, JSON with bare keys / single quotes, <key>value</key> children."""
    import html
    raw = html.unescape(raw or "").strip()
    if not raw:
        return {}
    obj = _loads_loose(raw)
    if isinstance(obj, dict):
        return obj
    m = re.search(r"\{.*\}", raw, re.S)
    if m:
        fixed = re.sub(r"([{,]\s*)([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1"\2":', m.group(0))
        obj = _loads_loose(fixed)
        if isinstance(obj, dict):
            return obj
        try:
            import ast
            obj = ast.literal_eval(_py_src(fixed))
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
    kids = {k: _value(v) for k, v in _CHILD_TAG.findall(raw)}
    return kids
_PY_CALL = re.compile(r"(?<![A-Za-z0-9_.])(" + _TOOL_NAME + r")\s*\(")
_XML_TAG = re.compile(r"<(" + _TOOL_NAME + r")((?:\s+[A-Za-z_][A-Za-z0-9_]*\s*=)[^<>]*?)?\s*/?>", re.S)
_ATTR = re.compile(r"([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(\"(?:[^\"\\]|\\.)*\"|'(?:[^'\\]|\\.)*'|\{.*?\}(?=\s+[A-Za-z_]\w*\s*=|\s*/?>|\s*$)|\[.*?\](?=\s+[A-Za-z_]\w*\s*=|\s*/?>|\s*$)|[^\s/>]+)", re.S)


def _value(v: str):
    v = v.strip()
    if len(v) >= 2 and v[0] == v[-1] and v[0] in "\"'":
        inner = v[1:-1]
        try:
            return json.loads(v) if v[0] == '"' else inner
        except ValueError:
            return inner
    low = v.lower()
    if low in ("true", "false"):
        return low == "true"
    if low in ("null", "none"):
        return None
    try:
        return json.loads(v)
    except ValueError:
        pass
    try:
        import ast
        return ast.literal_eval(v)
    except Exception:
        return v


def _balanced(text: str, i: int):
    """text[i] == '(' -> index of the matching ')' (quote-aware), or len(text)."""
    depth, q, k = 0, None, i
    while k < len(text):
        ch = text[k]
        if q:
            if ch == "\\":
                k += 1
            elif ch == q:
                q = None
        elif ch in "\"'":
            q = ch
        elif ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth -= 1
            if depth == 0:
                return k
        k += 1
    return len(text)


def _py_args(src: str):
    """'a="x", b=2, c=true' -> dict, via ast with JSON literals mapped to Python ones."""
    import ast
    try:
        tree = ast.parse("f(" + src + ")", mode="eval")
        out = {}
        for kw in tree.body.keywords:
            if kw.arg is None:
                continue
            try:
                out[kw.arg] = ast.literal_eval(kw.value)
            except Exception:
                seg = ast.get_source_segment("f(" + src + ")", kw.value) or ""
                out[kw.arg] = _value(seg)
        return out
    except SyntaxError:
        return {m.group(1): _value(m.group(2)) for m in _ATTR.finditer(src)}


def _py_src(src: str) -> str:
    return re.sub(r"\b(true|false|null)\b", lambda m: {"true": "True", "false": "False", "null": "None"}[m.group(1)], src)


def _scan_calls(text: str):
    """All non-<tool_call> formats, in order of appearance."""
    found = []
    import html
    taken = []
    for m in _FUNC_TAG.finditer(text):
        raw = m.group(2)
        if raw[:1] in "\"'":
            raw = html.unescape(raw[1:-1])
        found.append((m.start(), {"name": m.group(1), "arguments": _args_block(raw)}))
        taken.append((m.start(), m.end()))
    for m in _FUNC_NAME_TAG.finditer(text):
        if any(a <= m.start() < b for a, b in taken):
            continue
        found.append((m.start(), {"name": m.group(1), "arguments": _args_block(m.group(2) or "")}))
        taken.append((m.start(), m.end()))
    for m in _PY_CALL.finditer(text):
        if any(a <= m.start() < b for a, b in taken):
            continue
        close = _balanced(text, m.end() - 1)
        found.append((m.start(), {"name": m.group(1), "arguments": _py_args(_py_src(text[m.end():close]))}))
        taken.append((m.start(), close + 1))
    for m in _XML_TAG.finditer(text):
        if any(a <= m.start() < b for a, b in taken):
            continue
        attrs = m.group(2) or ""
        found.append((m.start(), {"name": m.group(1),
                                  "arguments": {a.group(1): _value(html.unescape(a.group(2))) for a in _ATTR.finditer(attrs)}}))
    if not found:
        for m in re.finditer(r"\{\s*\"name\"\s*:", text):
            obj = _loads_loose(text[m.start():])
            if isinstance(obj, dict) and isinstance(obj.get("name"), str):
                found.append((m.start(), obj))
                break
    return [c for _, c in sorted(found, key=lambda x: x[0])]


def parse_output(text: str):
    """-> (visible_content, [ {name, arguments} ])"""
    if "</think>" in text:
        text = text.split("</think>")[-1]
    text = text.replace("<think>", "").strip()
    calls = []
    for m in _TOOL_CALL.finditer(text):
        obj = _loads_loose(m.group(1))
        if isinstance(obj, dict) and isinstance(obj.get("name"), str):
            calls.append(obj)
        else:
            calls.extend(_scan_calls(m.group(1)))
    if not calls:
        calls = _scan_calls(text)
    out = []
    for c in calls:
        args = c.get("arguments", c.get("parameters", {}))
        if isinstance(args, str):
            args = _loads_loose(args) or {}
        out.append({"name": c["name"].strip(), "arguments": args if isinstance(args, dict) else {}})
    content = _TOOL_CALL.sub("", text).strip()
    return content, out


def fix_name(name: str, allowed: set[str]) -> str:
    if name in allowed:
        return name
    for prefix in ("functions.", "function.", "tools."):
        if name.startswith(prefix) and name[len(prefix):] in allowed:
            return name[len(prefix):]
    tail = [a for a in allowed if a.split(".", 1)[-1] == name or a.replace(".", "_") == name]
    return tail[0] if len(tail) == 1 else name


def coerce_args(args: dict, tool: dict | None) -> dict:
    """Cheap type fixes the schema clearly asks for (e.g. "26" -> 26, 0912 -> "0912"). Unknown keys are kept."""
    if not tool:
        return args
    props = (tool.get("parameters") or {}).get("properties") or {}
    out = {}
    for k, v in args.items():
        typ = (props.get(k) or {}).get("type")
        try:
            if typ == "integer" and isinstance(v, str) and re.fullmatch(r"-?\d+", v.strip()):
                v = int(v.strip())
            elif typ == "integer" and isinstance(v, float) and v.is_integer():
                v = int(v)
            elif typ == "number" and isinstance(v, str) and re.fullmatch(r"-?\d+(\.\d+)?", v.strip()):
                v = float(v) if "." in v else int(v)
            elif typ == "string" and isinstance(v, (int, float)) and not isinstance(v, bool):
                v = str(v)
            elif typ == "boolean" and isinstance(v, str) and v.lower() in {"true", "false"}:
                v = v.lower() == "true"
            elif typ == "array" and isinstance(v, str):
                v = [v]
        except (ValueError, AttributeError):
            pass
        out[k] = v
    return out


_PLACEHOLDER = re.compile(r"^\s*(\.\.\.|…|<[^>]*>|\{[^}]*\}|null|none|n/?a|unknown|tbd|\?+)?\s*$", re.I)


def drop_placeholders(args: dict, tool: dict | None) -> dict:
    """Drop optional arguments the model filled with nothing ("", "...", None): they would not match the gold call."""
    required = set(((tool or {}).get("parameters") or {}).get("required") or [])
    return {k: v for k, v in args.items()
            if k in required or not (v is None or (isinstance(v, str) and _PLACEHOLDER.match(v)) or v == [] or v == {})}


def cut_after_first_read(calls: list, catalog: dict) -> list:
    """FunReason-MT batches every call in one turn, e.g. gold_price.get_price + reminders.create for "if the price is
    below X, remind me" (Kaggle v8 train_0006). Run the batch only up to its first read call; the model sees the result
    and re-plans. Later calls that do not depend on it are simply emitted again in the next step."""
    for i, c in enumerate(calls):
        if (catalog.get(c["name"]) or {}).get("kind") == "read":
            return calls[: i + 1]
    return calls


# ----------------------------------------------------------------------------------------------- LLM
def _sdpa_expanded_kv(module, query, key, value, attention_mask, dropout=0.0, scaling=None, is_causal=None, **kwargs):
    """SDPA with K/V heads repeated up front, so the memory-efficient kernel stays eligible."""
    import torch
    groups = query.shape[1] // key.shape[1]
    if groups > 1:
        key = key.repeat_interleave(groups, dim=1)
        value = value.repeat_interleave(groups, dim=1)
    if attention_mask is not None and attention_mask.ndim == 4:
        attention_mask = attention_mask[:, :, :, : key.shape[-2]]
    if is_causal is None:
        is_causal = query.shape[2] > 1 and attention_mask is None and getattr(module, "is_causal", True)
    out = torch.nn.functional.scaled_dot_product_attention(
        query.contiguous(), key.contiguous(), value.contiguous(), attn_mask=attention_mask,
        dropout_p=dropout, scale=scaling, is_causal=is_causal)
    return out.transpose(1, 2).contiguous(), None


def memory_safe_attention() -> str:
    """GPUs without FlashAttention (T4, P100): SDPA with enable_gqa=True only runs the math kernel, which builds the full
    heads x seq x seq score matrix (several GB at 6-8K tokens -> OOM, and slow). Force the K/V-expanded path instead."""
    import torch
    if not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] >= 8:
        return "sdpa"
    try:
        import transformers.integrations.sdpa_attention as sdpa_mod
        if hasattr(sdpa_mod, "use_gqa_in_sdpa"):
            sdpa_mod.use_gqa_in_sdpa = lambda *a, **k: False
            return "sdpa"
    except ImportError:
        pass
    try:
        import transformers
        mask_iface = getattr(transformers, "AttentionMaskInterface", None)
        if mask_iface is None:
            from transformers.masking_utils import AttentionMaskInterface as mask_iface
        transformers.AttentionInterface.register("sdpa_expanded_kv", _sdpa_expanded_kv)
        mask_iface.register("sdpa_expanded_kv", mask_iface()["sdpa"])
        return "sdpa_expanded_kv"
    except Exception as exc:  # last resort: plain sdpa
        print(f"[llm] không đăng ký được attention tiết kiệm bộ nhớ: {exc}", flush=True)
        return "sdpa"


class LLM:
    def __init__(self, model_dir: Path, device: str, max_context: int = 16384):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch, self.device, self.max_context = torch, device, max_context
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        dtype = gpu_dtype()
        attention = memory_safe_attention()
        print(f"[llm] attention={attention}", flush=True)
        self.model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=dtype, attn_implementation=attention).to(device).eval()
        gc_ = self.model.generation_config
        gc_.do_sample, gc_.temperature, gc_.top_p, gc_.top_k = False, None, None, None
        self.eos = [i for i in {self.tok.convert_tokens_to_ids("<|im_end|>"), self.tok.eos_token_id,
                                self.tok.convert_tokens_to_ids("<|endoftext|>")} if isinstance(i, int) and i >= 0]
        self.has_think = "<think>" in (self.tok.chat_template or "") and "Instruct-2507" not in str(model_dir)
        self.params = sum(p.numel() for p in self.model.parameters())
        print(f"[llm] {model_dir} params={self.params / 1e9:.2f}B dtype={dtype}", flush=True)

    def render(self, messages, tools, think: bool) -> str:
        prompt = self.tok.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
        # FunReason-MT always opens a long <think> block (Kaggle v4: 616/634 samples ran out of tokens mid-thought).
        # Without --think, prefill an empty think block (Qwen3 non-thinking convention) so it answers directly.
        return prompt + self.render_suffix(think)

    def render_suffix(self, think: bool) -> str:
        if not getattr(self, "has_think", True):
            return ""
        return "<think>\n" if think else "<think>\n\n</think>\n\n"

    def think_end_ids(self):
        i = self.tok.convert_tokens_to_ids("</think>")
        return [i] if isinstance(i, int) and i >= 0 else []

    def generate_name(self, prompt: str, names: list[str]) -> str:
        """Constrained decoding of a tool name: only continuations that stay a prefix of one of `names` (followed by the
        closing quote) are allowed. Returns the chosen name."""
        import torch
        from transformers import LogitsProcessor, LogitsProcessorList, StoppingCriteria, StoppingCriteriaList
        tok, targets = self.tok, [n + '"' for n in names]

        def make(start):
            class OnlyNames(LogitsProcessor):
                def __call__(self, input_ids, scores):
                    done = tok.decode(input_ids[0, start:], skip_special_tokens=True)
                    live = [t for t in targets if t.startswith(done)]
                    allowed = set()
                    for t in live:  # the canonical next token of every still-possible name guarantees progress
                        rest = t[len(done):]
                        if rest:
                            allowed.add(tok(rest, add_special_tokens=False)["input_ids"][0])
                    for i in torch.topk(scores[0], 200).indices.tolist():
                        piece = tok.decode([i])
                        if piece and any(t.startswith(done + piece) for t in live):
                            allowed.add(i)
                    mask = torch.full_like(scores, float("-inf"))
                    idx = torch.tensor(sorted(allowed), device=scores.device, dtype=torch.long)
                    mask[0, idx] = scores[0, idx]
                    return mask

            class QuoteClosed(StoppingCriteria):
                def __call__(self, input_ids, scores, **kwargs):
                    done = tok.decode(input_ids[0, start:], skip_special_tokens=True)
                    return torch.full((input_ids.shape[0],), '"' in done, dtype=torch.bool, device=input_ids.device)

            return {"logits_processor": LogitsProcessorList([OnlyNames()]),
                    "stopping_criteria": StoppingCriteriaList([QuoteClosed()])}

        out = self.generate(prompt, 32, extra=make)
        return out.split('"')[0].strip()

    def tool_end_ids(self):
        i = self.tok.convert_tokens_to_ids("</tool_call>")
        return [i] if isinstance(i, int) and i >= 0 else []

    def count(self, text: str) -> int:
        return len(self.tok(text, add_special_tokens=False)["input_ids"])

    def reset(self):
        """Start of a sample: drop the KV cache (tools/system prompt differ between samples)."""
        self.cache, self.cache_ids = None, []

    def _stop_without_tool_call(self, start: int, grace: int, head_prefix: str = ""):
        """Stop a plain-text answer after `grace` tokens: only tool calls are scored, the closing text is not."""
        import torch
        from transformers import StoppingCriteria, StoppingCriteriaList
        call_id = self.tok.convert_tokens_to_ids("<tool_call>")
        think_id = self.tok.convert_tokens_to_ids("<think>")

        tok = self.tok

        class NoToolCall(StoppingCriteria):
            def __call__(self, input_ids, scores, **kwargs):
                gen = input_ids[0, start:]
                stop = (gen.shape[0] >= grace and not bool((gen == call_id).any()) and not bool((gen == think_id).any()))
                if stop:
                    head = head_prefix + tok.decode(gen[:grace], skip_special_tokens=True)
                    stop = looks_like_prose(head)
                return torch.full((input_ids.shape[0],), stop, dtype=torch.bool, device=input_ids.device)

        return StoppingCriteriaList([NoToolCall()])

    def generate(self, prompt: str, max_new_tokens: int, text_grace: int = 0, stop_ids=(), no_think: bool = False,
                 head_prefix: str = "", extra=None) -> str:
        """Greedy decode. Within a sample, the KV cache of the longest common token prefix with the previous step is reused,
        so each agent step only prefills the new tool result instead of the whole 6-8K-token prompt."""
        ids = self.tok(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"].to(self.device)
        new = ids[0].tolist()
        kwargs = {}
        cache = getattr(self, "cache", None)
        if cache is not None:
            old = self.cache_ids
            n, lim = 0, min(len(old), len(new) - 1)
            while n < lim and old[n] == new[n]:
                n += 1
            if n > 0:
                cache.crop(n)
                kwargs["past_key_values"] = cache
        if text_grace:
            kwargs["stopping_criteria"] = self._stop_without_tool_call(len(new), text_grace, head_prefix)
        if no_think:
            bad = [i for i in (self.tok.convert_tokens_to_ids("<think>"), self.tok.convert_tokens_to_ids("</think>"))
                   if isinstance(i, int) and i >= 0]
            if bad:
                kwargs["bad_words_ids"] = [[i] for i in bad]
        kwargs.update(extra(len(new)) if extra else {})
        with self.torch.inference_mode():
            out = self.model.generate(input_ids=ids, attention_mask=self.torch.ones_like(ids),
                                      max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
                                      eos_token_id=self.eos + list(stop_ids), pad_token_id=self.tok.pad_token_id or self.eos[0],
                                      return_dict_in_generate=True, **kwargs)
        seq = out.sequences[0]
        self.cache = out.past_key_values
        self.cache_ids = seq[: self.cache.get_seq_length()].tolist()
        return self.tok.decode(seq[len(new):], skip_special_tokens=True)


def fit_context(llm: LLM, messages, tools, think, max_new_tokens):
    """Shrink the longest tool outputs until prompt + reserved output fits the 16K context."""
    budget = llm.max_context - max_new_tokens
    prompt = llm.render(messages, tools, think)
    n = llm.count(prompt)
    while n > budget:
        tool_idx = [i for i, m in enumerate(messages) if m["role"] == "tool" and len(m["content"]) > 300]
        if not tool_idx:
            return None, n
        i = max(tool_idx, key=lambda k: len(messages[k]["content"]))
        c = messages[i]["content"]
        messages[i] = {"role": "tool", "content": c[: len(c) // 2] + " ...(cắt bớt)"}
        prompt = llm.render(messages, tools, think)
        n = llm.count(prompt)
    return prompt, n



# ----------------------------------------------------------------------------------------------- grounding / pruning / few-shot
_VI_DIGITS = {"không": "0", "một": "1", "mốt": "1", "hai": "2", "ba": "3", "bốn": "4", "tư": "4", "năm": "5", "lăm": "5",
              "sáu": "6", "bảy": "7", "bẩy": "7", "tám": "8", "chín": "9", "linh": "0", "lẻ": "0"}
PHONE_PARAMS = {"phone", "phone_number"}


def spoken_digits(text: str) -> str:
    """'không chín năm không ...' / '0950 ...' -> every maximal run of digits, joined by spaces."""
    runs, cur = [], ""
    for w in re.findall(r"\w+", unicodedata.normalize("NFC", str(text).lower())):
        if w.isdigit():
            cur += w
        elif w in _VI_DIGITS:
            cur += _VI_DIGITS[w]
        else:
            if cur:
                runs.append(cur)
            cur = ""
    if cur:
        runs.append(cur)
    return " ".join(runs)


def ungrounded_args(call: dict, catalog: dict, seen_text: str, said_digits: str) -> list[str]:
    """ID arguments (x-ref) must come from data the agent has already seen (tool results, history, context); phone numbers
    from that data or from what the user said. Kaggle v11: Qwen invented contact_123 / evt_12345 / 0987654321."""
    props = ((catalog.get(call["name"]) or {}).get("parameters") or {}).get("properties") or {}
    bad = []
    for k, v in (call.get("arguments") or {}).items():
        spec = props.get(k) or {}
        vals = v if isinstance(v, list) else [v]
        for x in vals:
            if not isinstance(x, (str, int)) or isinstance(x, bool):
                continue
            x = str(x)
            if spec.get("x-ref") and x not in seen_text:
                bad.append(f"{k}={x}")
            elif k in PHONE_PARAMS:
                digits = re.sub(r"\D", "", x)
                if digits and digits not in re.sub(r"\D", "", seen_text) and digits not in said_digits.replace(" ", ""):
                    bad.append(f"{k}={x}")
    return bad


def ref_hint(call: dict, catalog: dict, allowed: set) -> str:
    props = ((catalog.get(call["name"]) or {}).get("parameters") or {}).get("properties") or {}
    apps = {(spec.get("x-ref") or "").split(".")[0] for spec in props.values() if spec.get("x-ref")}
    if any(k in PHONE_PARAMS for k in (call.get("arguments") or {})):
        apps.add("contacts")
    reads = sorted(n for n in allowed if n.split(".")[0] in apps and (catalog.get(n) or {}).get("kind") == "read")
    return ", ".join(reads) or "tool tìm kiếm/liệt kê phù hợp"


def rare_optional_params(train_items, catalog: dict, max_rate: float = 0.1, min_uses: int = 3) -> set:
    """(tool, param) pairs that are optional in the schema and (almost) never set in the train gold calls."""
    use, present = Counter(), Counter()
    for it in train_items:
        for c in (it.get("label") or {}).get("tool_calls") or []:
            use[c["name"]] += 1
            for k in c.get("arguments") or {}:
                present[(c["name"], k)] += 1
    rare = set()
    for n, t in catalog.items():
        params = t.get("parameters") or {}
        req = set(params.get("required") or [])
        for k in params.get("properties") or {}:
            if k not in req and use[n] >= min_uses and present[(n, k)] / use[n] < max_rate:
                rare.add((n, k))
    return rare


def few_shot_block(train_items, catalog: dict, skip_ids=()) -> str:
    """Three solved train samples (taken past the first 200 so the local eval subset is not leaked)."""
    pool = [it for it in train_items[200:] if it["id"] not in set(skip_ids) and isinstance(it.get("audio_text"), str)
            and not it.get("history")]

    def names(it):
        return [c["name"] for c in it["label"]["tool_calls"]]

    picks = []
    tests = [
        lambda it: "nếu" in it["audio_text"].lower() and len(names(it)) == 1
        and (catalog.get(names(it)[0]) or {}).get("kind") == "read",
        lambda it: names(it)[:2] in (["contacts.search", "sms.send"], ["contacts.search", "phone.call"]),
        lambda it: len(names(it)) == 2 and names(it)[0] == "smart_home.get_status",
    ]
    for test in tests:
        hit = next((it for it in pool if it not in picks and test(it)), None)
        if hit:
            picks.append(hit)
    lines = ["# Ví dụ đã giải đúng (chỉ minh họa cách làm, dữ liệu là của người dùng khác)"]
    for it in picks:
        lines.append(f"Người dùng: {it['audio_text']}")
        res = it["label"].get("tool_results") or []
        for i, c in enumerate(it["label"]["tool_calls"]):
            lines.append(f"→ gọi {c['name']} {dumps(c.get('arguments') or {})}")
            if i < len(res):
                lines.append(f"← kết quả: {dumps(res[i])[:260]}")
        lines.append("→ kết thúc (không gọi thêm tool).")
        lines.append("")
    return "\n".join(lines) if picks else ""


FINISH_TOOL = {"type": "function", "function": {
    "name": "finish",
    "description": "Gọi khi yêu cầu của người dùng đã được xử lý xong, hoặc khi không có ứng dụng nào xử lý được yêu cầu "
                   "(khi đó không gọi tool nào khác). Không có tham số.",
    "parameters": {"type": "object", "properties": {}, "required": []}}}


def run_agent(item, env, llm: LLM, catalog: dict, system_prompt: str, transcripts: dict, args) -> dict:
    tools = tool_specs(catalog, item)
    allowed = {t["function"]["name"] for t in tools}
    constrained = bool(getattr(args, "constrained", False))
    if constrained:
        tools = tools + [FINISH_TOOL]
    messages = build_messages(item, system_prompt, transcripts, args.gold_transcript)
    trace = {"request": messages[-1]["content"], "steps": []}
    seen_text = dumps(item.get("context") or {}) + dumps(item.get("history") or [])
    said_digits = spoken_digits(" ".join(m["content"] for m in messages if m["role"] == "user"))
    rare = getattr(args, "rare_params", None) or set()
    seen: dict[str, dict] = {}
    last_fail, fail_streak = None, 0
    if hasattr(llm, "reset"):
        llm.reset()
    t0 = time.time()
    for step in range(args.max_steps):
        if args.time_budget and time.time() - t0 > args.time_budget:
            trace["stop"] = "time_budget"
            break
        prompt, n_tokens = fit_context(llm, messages, tools, args.think, args.max_new_tokens)
        if prompt is None:
            trace["stop"] = f"context_overflow({n_tokens})"
            break
        g0 = time.time()
        thought, prefix = "", ""
        if getattr(args, "think_budget", 0):
            # FunReason-MT reasons before every call; with an empty <think> it reasons in plain text instead (Kaggle v5:
            # 634/634 empty). Budget forcing: think up to N tokens, close the block ourselves, then decode the answer.
            p_think = prompt[: -len(llm.render_suffix(False))] + llm.render_suffix(True)
            thought = llm.generate(p_think, args.think_budget, stop_ids=llm.think_end_ids())
            closed = "</think>" in thought
            thought = thought.split("</think>")[0].rstrip()
            prompt = p_think + thought + "\n</think>\n\n"
            if not closed:
                # Kaggle v8: after a forced </think> the model restarted prose reasoning ("Okay, let's see...") in 6/12
                # samples and never emitted a call. Every call format it uses starts with "<", and so does <answer>.
                prefix = "<"
                prompt += prefix
        if constrained:
            # The answer is always one tool call; the model only picks the name (or `finish`) and writes the arguments.
            head = prompt[: len(prompt) - len(prefix)] if prefix else prompt
            head += '<tool_call>\n{"name": "'
            name = llm.generate_name(head, sorted(allowed) + ["finish"])
            if name == "finish" or name not in allowed:
                trace["steps"].append({"think": thought[-1500:], "raw": f"[constrained] {name}", "prompt_tokens": n_tokens,
                                       "seconds": round(time.time() - g0, 2)})
                trace["stop"] = "answer"
                break
            head += name + '", "arguments": '
            body = llm.generate(head, args.max_new_tokens, stop_ids=llm.tool_end_ids())
            raw = '<tool_call>\n{"name": "' + name + '", "arguments": ' + body.split("</tool_call>")[0] + "\n</tool_call>"
        else:
            if getattr(args, "force_call", False) and step == 0 and not prefix:
                prefix = "<tool_call>\n"
                prompt += prefix
            raw = prefix + llm.generate(prompt, args.max_new_tokens, text_grace=0 if args.think else args.text_grace,
                                        no_think=bool(getattr(args, "think_budget", 0)), head_prefix=prefix)
        content, calls = parse_output(raw)
        trace["steps"].append({"think": thought[-1500:], "raw": raw[-2000:], "prompt_tokens": n_tokens,
                               "seconds": round(time.time() - g0, 2)})
        if not calls:
            trace["stop"] = "answer"
            break
        for c in calls:
            c["name"] = fix_name(c["name"], allowed)
            c["arguments"] = drop_placeholders(coerce_args(c["arguments"], catalog.get(c["name"])), catalog.get(c["name"]))
            c["arguments"] = {k: v for k, v in c["arguments"].items() if (c["name"], k) not in rare}
        calls = cut_after_first_read(calls, catalog)
        messages.append({"role": "assistant", "content": content, "tool_calls": [
            {"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}} for c in calls]})
        stop = False
        for c in calls:
            key = c["name"] + dumps(c["arguments"])
            check = env.validate(c["name"], c["arguments"])  # schema check only: not logged, nothing executed
            if key in seen and seen[key].get("ok") is not False:
                result = {"ok": False, "error": "Call này đã được thực hiện thành công ở trên, không gọi lại. "
                                                "Hãy làm bước tiếp theo hoặc trả lời kết thúc."}
            elif getattr(args, "ground", True) and ungrounded_args(c, catalog, seen_text, said_digits):
                bad = ungrounded_args(c, catalog, seen_text, said_digits)
                result = {"ok": False, "error": f"Chưa thực thi: {', '.join(bad)} không có trong dữ liệu đã thấy. Không tự đặt ID "
                                                f"hay số điện thoại; hãy gọi {ref_hint(c, catalog, allowed)} trước để lấy giá trị đúng."}
            elif not check.get("ok"):
                # A rejected env.call would still be logged and scored as an unmatched call; let the model fix it.
                result = {"ok": False, "error": f"Chưa thực thi, sai schema: {check.get('error')}. Sửa lại tham số "
                                                "hoặc dùng sys.ask_missing_info nếu thiếu thông tin."}
            else:
                result = env.call(c["name"], c["arguments"])
                seen[key] = result
                seen_text += dumps(result)
            messages.append({"role": "tool", "content": dumps(result)[: args.max_tool_chars]})
            fail = key + dumps(result) if result.get("ok") is False else None
            fail_streak = fail_streak + 1 if fail and fail == last_fail else (1 if fail else 0)
            last_fail = fail
            if c["name"] == "sys.ask_missing_info" and result.get("ok") and result.get("waiting_for_user"):
                stop, trace["stop"] = True, "ask_missing_info"
                break
            if fail_streak >= 3:
                stop, trace["stop"] = True, "repeated_failure"
                break
        if stop:
            break
    else:
        trace["stop"] = "max_steps"
    trace["seconds"] = round(time.time() - t0, 2)
    return trace


# ----------------------------------------------------------------------------------------------- local scoring (Train only)
def _nfc_tokens(s) -> list[str]:
    return unicodedata.normalize("NFC", str(s).lower()).split()


def token_f1(a, b) -> float:
    ta, tb = _nfc_tokens(a), _nfc_tokens(b)
    if not ta and not tb:
        return 1.0
    common = sum((Counter(ta) & Counter(tb)).values())
    if common == 0:
        return 0.0
    p, r = common / len(ta), common / len(tb)
    return 2 * p * r / (p + r)


def fuzzy_params(catalog: dict) -> dict:
    return {n: {k for k, v in ((t.get("parameters") or {}).get("properties") or {}).items() if v.get("x-match") == "fuzzy"}
            for n, t in catalog.items()}


def calls_match(p, g, fuzzy: dict) -> bool:
    if p.get("name") != g.get("name"):
        return False
    pa, ga = p.get("arguments") or {}, g.get("arguments") or {}
    fz = fuzzy.get(g["name"], set())
    if set(pa) != set(ga):
        return False
    for k, gv in ga.items():
        if k in fz:
            if token_f1(pa[k], gv) <= 0.6:
                return False
        elif pa[k] != gv:
            return False
    return True


def score_sample(pred_calls, pred_hash, gold_calls, gold_hash, fuzzy) -> float:
    """Our reading of the brief: empty gold -> 1 iff empty prediction; else 0 on state mismatch, otherwise F1(P, R)."""
    if not gold_calls:
        return float(not pred_calls)
    if pred_hash != gold_hash:
        return 0.0
    used, m = set(), 0
    for p in pred_calls:
        for j, g in enumerate(gold_calls):
            if j not in used and calls_match(p, g, fuzzy):
                used.add(j)
                m += 1
                break
    if m == 0:
        return 0.0
    prec, rec = m / len(pred_calls), m / len(gold_calls)
    return 2 * prec * rec / (prec + rec)


# ----------------------------------------------------------------------------------------------- main
def parse_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="public_test", choices=("public_test", "train", "private_test"))
    ap.add_argument("--data-root", help="thư mục chứa <split>/ (tự dò trong /kaggle/input nếu bỏ trống)")
    ap.add_argument("--asr-model", help="thư mục Whisper zaic/ (tự dò nếu bỏ trống)")
    ap.add_argument("--llm-model", default=DEFAULT_LLM, help="thư mục local hoặc repo HF")
    ap.add_argument("--llm-download-dir", default="", help="mặc định /tmp/<tên repo>")
    ap.add_argument("--prompt-file", help="file system prompt (UTF-8); mặc định dùng prompt có sẵn")
    ap.add_argument("--out", default="submission.jsonl")
    ap.add_argument("--trace", default="traces.jsonl")
    ap.add_argument("--transcripts", default="transcripts_{split}.json", help="cache ASR")
    ap.add_argument("--limit", type=int, default=0, help="chỉ chạy N mẫu đầu (0 = tất cả)")
    ap.add_argument("--ids", nargs="*", default=[])
    ap.add_argument("--max-steps", type=int, default=10)
    ap.add_argument("--max-new-tokens", type=int, default=512)
    ap.add_argument("--max-tool-chars", type=int, default=6000)
    ap.add_argument("--time-budget", type=float, default=60.0, help="giây mỗi mẫu cho vòng LLM (0 = không giới hạn)")
    ap.add_argument("--think-budget", type=int, default=0,
                    help="nghĩ tối đa N token trong <think>, rồi tự đóng </think> và sinh tool call (0 = tắt)")
    ap.add_argument("--no-few-shot", dest="few_shot", action="store_false", help="tắt 3 ví dụ lấy từ train")
    ap.add_argument("--no-prune", dest="prune", action="store_false", help="giữ cả tham số tùy chọn hiếm dùng")
    ap.add_argument("--no-ground", dest="ground", action="store_false", help="không chặn ID/số điện thoại tự bịa")
    ap.add_argument("--constrained", action="store_true",
                    help="mỗi bước là đúng một tool call: tên tool được giải mã có ràng buộc (hoặc `finish` để kết thúc)")
    ap.add_argument("--force-call", action="store_true", help="bước đầu tiên luôn bắt đầu bằng <tool_call>")
    ap.add_argument("--think", action="store_true", help="mở <think> như model card (chậm hơn nhiều)")
    ap.add_argument("--text-grace", type=int, default=20,
                    help="dừng câu trả lời chữ sau N token nếu chưa có <tool_call> (0 = tắt)")
    ap.add_argument("--shard", default="0/1", help="i/n: chỉ chạy mẫu thứ i, i+n, ... (chia cho nhiều GPU)")
    ap.add_argument("--prepare-only", action="store_true", help="chỉ chạy ASR + tải LLM rồi thoát")
    ap.add_argument("--asr-batch", type=int, default=16)
    ap.add_argument("--gold-transcript", action="store_true", help="Train: dùng audio_text thay ASR")
    ap.add_argument("--eval", action="store_true", help="Train: chấm điểm cục bộ với nhãn")
    ap.add_argument("--asr-device", default="cuda:0")
    ap.add_argument("--llm-device", default="cuda:0")
    return ap.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    import torch
    if not torch.cuda.is_available():
        args.asr_device = args.llm_device = "cpu"
    seed_everything()
    root = find_data_root(args.split, args.data_root)
    split_dir = root / args.split
    print(f"[data] root={root}", flush=True)
    sys.path.insert(0, str(split_dir))
    from zaic_tools import ToolEnv  # noqa: E402

    items = [json.loads(l) for l in open(split_dir / f"{args.split}.jsonl", encoding="utf-8") if l.strip()]
    if args.ids:
        items = [it for it in items if it["id"] in set(args.ids)]
    if args.limit:
        items = items[: args.limit]
    shard_i, shard_n = (int(x) for x in args.shard.split("/"))
    all_items, items = items, items[shard_i::shard_n]
    print(f"[data] {len(items)} mẫu (shard {args.shard})", flush=True)
    catalog = load_catalog(split_dir, args.split)
    system_prompt = (open(args.prompt_file, encoding="utf-8").read() if args.prompt_file else DEFAULT_SYSTEM_PROMPT)

    transcripts = {}
    if not (args.gold_transcript and args.split == "train"):
        asr_dir = find_asr_model(args.asr_model)
        transcripts = build_transcripts(all_items if args.prepare_only else items, split_dir, asr_dir, Path(args.transcripts.format(split=args.split)),
                                        args.asr_device, args.asr_batch)

    llm_dir = resolve_llm(args.llm_model, args.llm_download_dir)
    if args.prepare_only:
        print(f"[prepare] transcripts={len(transcripts)} llm={llm_dir}", flush=True)
        return
    llm = LLM(llm_dir, args.llm_device)
    fuzzy = fuzzy_params(catalog)
    train_items = []
    for cand in {root / "train" / "train.jsonl", split_dir.parent / "train" / "train.jsonl"}:
        if cand.is_file():
            train_items = [json.loads(l) for l in open(cand, encoding="utf-8") if l.strip()]
            break
    args.rare_params = rare_optional_params(train_items, catalog) if (train_items and args.prune) else set()
    if train_items and args.few_shot:
        eval_ids = [it["id"] for it in items] if args.split == "train" else []
        system_prompt = system_prompt.rstrip() + "\n\n" + few_shot_block(train_items, catalog, eval_ids)
    print(f"[agent] rare optional params={len(args.rare_params)} few_shot={args.few_shot} ground={args.ground}", flush=True)
    scores = []
    t_start = time.time()
    with open(args.out, "w", encoding="utf-8") as fout, open(args.trace, "w", encoding="utf-8") as ftr:
        for n, item in enumerate(items, 1):
            env = ToolEnv(args.split, item["id"], root=root)  # exactly one environment per sample
            try:
                trace = run_agent(item, env, llm, catalog, system_prompt, transcripts, args)
            except Exception as exc:  # keep the calls already made; never re-create the env
                trace = {"stop": f"error: {type(exc).__name__}: {exc}"}
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            row = {"id": item["id"], "tool_calls": env.get_tool_calls(), "state_hash": env.get_state_hash()}
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            fout.flush()
            msg = f"[{n}/{len(items)}] {item['id']} calls={len(row['tool_calls'])} stop={trace.get('stop')} {trace.get('seconds', 0)}s"
            if args.eval and "label" in item:
                gold_env = ToolEnv(args.split, item["id"], root=root)  # scorer-side replay of the gold label only
                for c in item["label"]["tool_calls"]:
                    gold_env.call(c["name"], c.get("arguments") or {})
                s = score_sample(row["tool_calls"], row["state_hash"], item["label"]["tool_calls"],
                                 gold_env.get_state_hash(), fuzzy)
                scores.append(s)
                trace["score"], trace["gold"] = s, item["label"]["tool_calls"]
                msg += f" score={s:.2f} mean={sum(scores) / len(scores):.4f}"
            print(msg, flush=True)
            # What the ASR heard (or the gold transcript with --gold-transcript), and what the agent called
            print(f"    nghe: {trace.get('request', '')[:300]}", flush=True)
            if args.eval and "audio_text" in item and not args.gold_transcript:
                gold_text = item["audio_text"] if isinstance(item["audio_text"], str) else item["audio_text"].get(item["audio"], "")
                print(f"    gốc : {str(gold_text)[:300]}", flush=True)
            print("    gọi : " + "; ".join(c["name"] + dumps(c.get("arguments") or {})[:120] for c in row["tool_calls"]), flush=True)
            ftr.write(json.dumps({"id": item["id"], **trace, "tool_calls": row["tool_calls"]}, ensure_ascii=False) + "\n")
            ftr.flush()
    total = time.time() - t_start
    print(f"[done] {len(items)} dòng -> {args.out}  ({total / max(len(items), 1):.1f}s/mẫu)", flush=True)
    if scores:
        print(f"[eval] {args.split} n={len(scores)} score={sum(scores) / len(scores):.4f}", flush=True)


if __name__ == "__main__":
    main()

## 1) Chạy thử 5 mẫu public_test

In [ ]:
!python lifemate_infer.py --split public_test --limit 5 --prompt-file system_prompt.txt --out smoke.jsonl --trace smoke_traces.jsonl
import json
_smoke = [json.loads(l) for l in open('smoke.jsonl', encoding='utf-8')]
_n = sum(len(r['tool_calls']) for r in _smoke)
print('smoke:', len(_smoke), 'mẫu,', _n, 'tool call')
for r in _smoke: print(r['id'], [c['name'] for c in r['tool_calls']])
assert _n > 0, 'Smoke test không ra tool call nào -> dừng, không chạy toàn bộ (xem smoke_traces.jsonl)'

## 2) So sánh cấu hình trên Train, chọn cấu hình tốt nhất
`--eval` chấm theo cách đọc công thức của đề (state_hash khớp → F1 của tool calls). Thêm `--gold-transcript` để bỏ qua ASR, chỉ đo LLM.
Lưu ý: nếu ASR `zaic` đã fine-tune trên train audio thì điểm ASR trên Train sẽ lạc quan.

In [ ]:
# So sánh cấu hình LLM trên N mẫu train (transcript gốc, chỉ đo LLM), mỗi GPU một hàng đợi; chọn cấu hình tốt nhất cho bước 3.
import json, os, re, subprocess, sys, time, torch
MIN_SCORE = 0.25   # chỉ chạy toàn bộ public_test nếu điểm train của cấu hình tốt nhất >= ngưỡng này
N_EVAL, MAX_SEC = 30, 75   # MAX_SEC: s/mẫu tối đa trên T4 để cả public_test vẫn xong trong giới hạn 12h của Kaggle
TB = ["--time-budget", "150"]
CONFIGS = {"qwen_fewshot": TB, "qwen_nofewshot": ["--no-few-shot"] + TB}
QUEUES = [["qwen_fewshot"], ["qwen_nofewshot"]]
n_gpu = max(torch.cuda.device_count(), 1)
if n_gpu == 1:
    QUEUES = [sum(QUEUES, [])]
def run_queue(gpu, names):
    cmds = "; ".join(" ".join([sys.executable, "lifemate_infer.py", "--split", "train", "--limit", str(N_EVAL), "--eval",
                                 "--gold-transcript", "--prompt-file", "system_prompt.txt", "--out", f"exp_{n}.jsonl",
                                 "--trace", f"exp_{n}_traces.jsonl"] + CONFIGS[n]) + f" > exp_{n}.log 2>&1" for n in names)
    return subprocess.Popen(cmds, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
procs = [run_queue(g, q) for g, q in enumerate(QUEUES)]
for p in procs:
    p.wait()
results = {}
for n in CONFIGS:
    log = open(f"exp_{n}.log", encoding="utf-8").read() if os.path.exists(f"exp_{n}.log") else ""
    sc = re.findall(r"\[eval\] train n=\d+ score=([0-9.]+)", log)
    sp = re.findall(r"\(([0-9.]+)s/mẫu\)", log)
    results[n] = (float(sc[-1]) if sc else -1.0, float(sp[-1]) if sp else 1e9)
    print(f"{n:16s} score={results[n][0]:.4f}  {results[n][1]:.1f}s/mẫu", flush=True)
    if not sc:
        print(log[-2000:])
ok = {n: r for n, r in results.items() if r[1] <= MAX_SEC} or results
BEST = max(ok, key=lambda n: (ok[n][0], -ok[n][1]))
BEST_FLAGS = CONFIGS[BEST]
print("BEST =", BEST, BEST_FLAGS)

## Chẩn đoán: tool call dự đoán so với nhãn (in thẳng vào log)

In [ ]:
import json, sys
sys.path.insert(0, ".")
import lifemate_infer as L
root = L.find_data_root("train", None)
gold = {}
for l in open(root / "train" / "train.jsonl", encoding="utf-8"):
    if l.strip():
        it = json.loads(l); gold[it["id"]] = it
def short(calls):
    return [c["name"] + json.dumps(c.get("arguments") or {}, ensure_ascii=False)[:160] for c in calls]
for n in CONFIGS:
    preds = {json.loads(l)["id"]: json.loads(l) for l in open(f"exp_{n}.jsonl", encoding="utf-8")}
    for l in open(f"exp_{n}_traces.jsonl", encoding="utf-8"):
        t = json.loads(l); it = gold[t["id"]]
        print("=" * 100)
        print(f"[{n}] {t['id']} stop={t.get('stop')} steps={len(t['steps'])} {t.get('seconds')}s")
        print("  REQ :", t["request"][:200])
        print("  GOLD:", short(it["label"]["tool_calls"]))
        print("  PRED:", short(preds[t["id"]]["tool_calls"]))
        for k, st in enumerate(t["steps"]):
            print(f"  step{k} think_tail:", repr(st.get("think", "")[-300:]))
            print(f"  step{k} raw:", repr(st["raw"][:400]))
print(f"BEST={BEST} score={results[BEST][0]:.4f} (ngưỡng {MIN_SCORE})")
assert results[BEST][0] >= MIN_SCORE, "Điểm train dưới ngưỡng -> dừng, không chạy toàn bộ public_test (xem chẩn đoán ở trên)"

## 3) Chạy toàn bộ public_test → submission.jsonl
ASR chạy một lần cho mọi mẫu, sau đó mỗi GPU chạy một nửa số mẫu (mỗi GPU một bản LLM), cuối cùng ghép lại theo đúng thứ tự.

In [ ]:
import json, os, subprocess, sys, time, torch
SPLIT, W = "public_test", "/kaggle/working"
base = [sys.executable, "lifemate_infer.py", "--split", SPLIT, "--prompt-file", "system_prompt.txt"] + BEST_FLAGS
subprocess.run(base + ["--prepare-only"], check=True)
n_gpu = max(torch.cuda.device_count(), 1)
procs = []
for i in range(n_gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
    log = open(f"{W}/shard_{i}.log", "w")
    procs.append(subprocess.Popen(base + ["--shard", f"{i}/{n_gpu}", "--out", f"{W}/sub_{i}.jsonl",
                                          "--trace", f"{W}/traces_{i}.jsonl"], env=env, stdout=log, stderr=subprocess.STDOUT))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    done = sum(sum(1 for _ in open(f"{W}/sub_{i}.jsonl", encoding="utf-8")) if os.path.exists(f"{W}/sub_{i}.jsonl") else 0
               for i in range(n_gpu))
    print(f"{(time.time() - t0) / 60:.0f} phút: {done} mẫu xong", flush=True)
print("exit codes", [p.returncode for p in procs])
for i in range(n_gpu):
    print(f"--- shard {i} (cuối log)"); print(open(f"{W}/shard_{i}.log", encoding="utf-8").read()[-1500:])
import lifemate_infer as L
order = [json.loads(l)["id"] for l in open(L.find_data_root(SPLIT, None) / SPLIT / f"{SPLIT}.jsonl", encoding="utf-8") if l.strip()]
rows = {}
for i in range(n_gpu):
    for l in open(f"{W}/sub_{i}.jsonl", encoding="utf-8"):
        r = json.loads(l); rows[r["id"]] = r
with open(f"{W}/submission.jsonl", "w", encoding="utf-8") as f:
    for sid in order:
        if sid in rows:
            f.write(json.dumps(rows[sid], ensure_ascii=False) + "\n")
with open(f"{W}/traces.jsonl", "w", encoding="utf-8") as f:
    for i in range(n_gpu):
        f.write(open(f"{W}/traces_{i}.jsonl", encoding="utf-8").read())
print("submission.jsonl:", sum(1 for sid in order if sid in rows), "/", len(order), "dòng")

## 4) Kiểm tra file nộp bằng `zaic_tools --validate`

In [ ]:
import json, subprocess, sys
sys.path.insert(0, ".")
import lifemate_infer as L
root = L.find_data_root("public_test", None)
rows = [json.loads(l) for l in open("/kaggle/working/submission.jsonl", encoding="utf-8")]
n_items = sum(1 for l in open(root / "public_test" / "public_test.jsonl", encoding="utf-8") if l.strip())
print("lines", len(rows), "expected", n_items, "unique ids", len({r["id"] for r in rows}))
print("empty tool_calls:", sum(not r["tool_calls"] for r in rows),
      "| ask_missing_info:", sum(any(c["name"] == "sys.ask_missing_info" for c in r["tool_calls"]) for r in rows))
print(subprocess.run([sys.executable, "-B", "-m", "zaic_tools", "--split", "public_test", "--root", str(root),
                      "--validate", "/kaggle/working/submission.jsonl"],
                     cwd=root / "public_test", capture_output=True, text=True).stdout[-3000:])